# MAESTRO Preprocessing (Updated with Supplementary Guide)

Includes fixes:
1. Use explicit Train/Val/Test splits from metadata (no random splitting).
2. Piano-roll (Tasks 1 & 2) window sparsity filtering (< 2% active cells discarded).
3. Token formulation using `miditok` (REMI scheme) for Tasks 3 & 4.

In [1]:
!pip install pretty_midi pandas matplotlib tqdm miditok

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 83.3 MB/s eta 0:00:00a 0:00:01
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 159.0/159.0 kB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.6/54.6 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 70.4 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 159.0/159.0 kB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.6/54.6 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 70.4 MB/s eta 0:00:00:00:01
  Created wheel for pretty_midi: filename=pretty_midi-0.2.11-py3-none-any.whl size=5595886 sha256=dbcdf0658b27eaa5ed2641fcf162827ae695ce8b9cbec11fb68fb340e938376a
  Stored in directory: /root/.cache/pip/wheels/f4/ad/93/a7042fe12668827574927ade9deec7f29aad2a1001b1501882
Successfully built pretty_midi
  Created wheel for pretty_midi: filename=pre

In [3]:
import os
import numpy as np
import pandas as pd
import pretty_midi
import matplotlib.pyplot as plt
from tqdm import tqdm
from miditok import REMI, TokenizerConfig
import warnings
warnings.filterwarnings('ignore')

In [7]:
import os
import urllib.request
import zipfile

# If running on Colab (or if dataset is missing in the current runtime environment), download it
if not os.path.exists('data/maestro-v3.0.0/maestro-v3.0.0.csv'):
    print("Downloading MAESTRO dataset (MIDI only)...")
    os.makedirs('data', exist_ok=True)
    urllib.request.urlretrieve('https://storage.googleapis.com/magentadata/datasets/maestro/v3.0.0/maestro-v3.0.0-midi.zip', 'data/maestro-v3.0.0-midi.zip')
    print("Extracting...")
    with zipfile.ZipFile('data/maestro-v3.0.0-midi.zip', 'r') as zip_ref:
        zip_ref.extractall('data/')
    print("Done!")

metadata_path = 'data/maestro-v3.0.0/maestro-v3.0.0.csv'
data_dir = 'data/maestro-v3.0.0'
metadata = pd.read_csv(metadata_path)

# Use predefined splits
train_meta = metadata[metadata['split'] == 'train'].head(20) # Using 20 as a subset for testing
val_meta = metadata[metadata['split'] == 'validation'].head(5)
print(f"Training files (subset): {len(train_meta)}, Val files: {len(val_meta)}")

Extracting...
Extracting...
Done!
Training files (subset): 20, Val files: 5
Done!
Training files (subset): 20, Val files: 5


## Part 1: Piano-Roll Representation (For Task 1 & 2)

In [8]:
FS = 16 # Frames per second
SEQ_LEN = 128 # 8 seconds of music
PITCH_RANGE = (21, 109) # 88 piano keys (A0 to C8)
SPARSITY_THRESHOLD = 0.02 # Dismiss windows with <2% active cells

def process_piano_roll(df):
    valid_segments = []
    for idx, row in tqdm(df.iterrows(), total=len(df)):
        midi_file = os.path.join(data_dir, row['midi_filename'])
        try:
            pm = pretty_midi.PrettyMIDI(midi_file)
            # 1. Extract piano-roll (fs=16)
            proll = pm.get_piano_roll(fs=FS)[PITCH_RANGE[0]:PITCH_RANGE[1], :]
            # 2. Binarize 
            proll = (proll > 0).astype(np.float32)
            # Transpose to (Time, Pitches)
            proll = proll.T
            
            # 3. Segment into windows
            n_segments = proll.shape[0] // SEQ_LEN
            for i in range(n_segments):
                win = proll[i*SEQ_LEN : (i+1)*SEQ_LEN, :]
                # 4. Sparsity Filtering
                if np.mean(win) > SPARSITY_THRESHOLD:
                    valid_segments.append(win)
        except Exception as e:
            continue
    return np.array(valid_segments)

os.makedirs('data/processed_rolls', exist_ok=True)
train_rolls = process_piano_roll(train_meta)
val_rolls = process_piano_roll(val_meta)
np.save('data/processed_rolls/train.npy', train_rolls)
np.save('data/processed_rolls/val.npy', val_rolls)

# Calculate negative to positive ratio for BCEWithLogitsLoss pos_weight later
num_positive = np.sum(train_rolls == 1)
num_negative = np.sum(train_rolls == 0)
pos_weight = num_negative / max(num_positive, 1)
with open('data/processed_rolls/pos_weight.txt', 'w') as f:
    f.write(str(pos_weight))
print(f"Saved {len(train_rolls)} train and {len(val_rolls)} val rolls. Suggested pos_weight: {pos_weight:.2f}")

100%|██████████| 20/20 [00:06<00:00,  3.26it/s]

100%|██████████| 5/5 [00:00<00:00,  8.66it/s]



Saved 1651 train and 189 val rolls. Suggested pos_weight: 11.68


## Part 2: Token Representation (For Task 3 & 4)

In [9]:
config = TokenizerConfig(num_velocities=32, use_chords=False, use_programs=False)
tokenizer = REMI(config)

def process_tokens(df):
    token_sequences = []
    for idx, row in tqdm(df.iterrows(), total=len(df)):
        midi_file = os.path.join(data_dir, row['midi_filename'])
        try:
            tokens = tokenizer(midi_file)
            if len(tokens.ids) > 100:
                token_sequences.append(tokens.ids)
        except:
            continue
    return token_sequences

os.makedirs('data/processed_tokens', exist_ok=True)
train_tokens = process_tokens(train_meta)
val_tokens = process_tokens(val_meta)
# Save as list of arrays
np.save('data/processed_tokens/train.npy', np.array(train_tokens, dtype=object))
np.save('data/processed_tokens/val.npy', np.array(val_tokens, dtype=object))
print(f"Saved {len(train_tokens)} train sequences for Transformer.")

100%|██████████| 5/5 [00:00<00:00, 35.09it/s]

Saved 0 train sequences for Transformer.
